# 03 - Compare layouts and movers

**Goal:** choose coil layout + magnet array together (Kuijpers 2023: pick the magnet pitch first,
then size the coils to it). Uses `sweep` to evaluate every combination.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from maglev_sim import *
from maglev_sim import plotting

In [ ]:
base = load_config("../configs/examples/single_tile_quick.yaml")
from maglev_sim.sweep import sweep
df = sweep(base, {
    "tile.params.pitch_mm": [20, 25, 30],
    "mover.params.size_mm": [6, 8, 10],
}, per_tile=3)
cols = ["tile.params.pitch_mm", "mover.params.size_mm", "mover_mass_g", "hover_ok_fraction",
        "lift_margin_worst", "hover_power_W_mean", "condition_worst"]
df[cols].sort_values("lift_margin_worst", ascending=False).round(2)

Coils must fit their pitch: the sweep reports warnings like overlapping coils.

In [ ]:
df[["tile.params.pitch_mm", "mover.params.size_mm", "warnings"]]

## Different mover types on the same tile

In [ ]:
movers = {
    "cube checkerboard 4x4": {"type": "cube_grid", "params": {"nx": 4, "ny": 4, "size_mm": 8, "pitch_mm": 16}},
    "disk grid 3x3":        {"type": "disk_grid", "params": {"nx": 3, "ny": 3, "diameter_mm": 12, "height_mm": 6, "pitch_mm": 18}},
    "2D Halbach n=5":       {"type": "halbach_2d", "params": {"n": 5, "size_mm": 8}},
}
rows = []
for name, m in movers.items():
    cfg = with_changes(base, {"mover": m})
    r = evaluate_design(build_system(cfg), per_tile=3, gaps_mm=[3])
    rows.append({"mover": name, **{k: r.summary[k] for k in
                 ["mover_mass_g", "hover_ok_fraction", "lift_margin_worst", "hover_power_W_mean", "condition_worst"]}})
import pandas as pd
pd.DataFrame(rows).round(2)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(16, 5))
for ax, (name, m) in zip(axs, movers.items()):
    plotting.plot_layout(build_system(with_changes(base, {"mover": m})), ax=ax)
    ax.set_title(name)

**Open question for the team:** the 2D Halbach from the proposal does worse here than a simple
checkerboard. Is that the magnet size, the pitch match, or the coil shape? This notebook is the
place to find out - try `halbach_2d` with different `size_mm` and coil pitches.